In [0]:

#tgt = target
#src = source

from pyspark.sql import functions as F
from delta.tables import DeltaTable

def write_to_gold(input_df, target_table, merge_condition, columns_to_update):

    final_df = (
        input_df
            .withColumn("created_timestamp",F.current_timestamp())
            .withColumn("updated_timestamp",F.current_timestamp())
    )
    
    if not spark.catalog.tableExists(target_table):
        (
            final_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
            )

    else:
        delta_table = DeltaTable.forName(spark, target_table)
        update_map = {column: f"src.{column}" for column in columns_to_update}
        update_map["updated_timestamp"] = "src.updated_timestamp"
        (
            delta_table.alias("tgt").merge(final_df.alias("src"), merge_condition)
            .whenMatchedUpdate(
                set = update_map
            )
            .whenNotMatchedInsertAll()
            .execute()
        )